# Notebook 02 — Transform, Segment, and Analyze Churn
## Contoso Banque — Churn Analysis Workshop

> **All data used in this notebook is entirely synthetic and fictional.**

---

## What This Notebook Does

This notebook reads the 5 raw Delta tables created by Notebook 01 and produces two curated analytical tables:

| Output | Description |
|---|---|
| `customer_360` | One row per customer — all features combined: demographics, balance metrics, product count, transaction activity, segments, and churn label |
| `churn_by_segment` | Aggregated churn KPIs grouped by segment (activity tier, balance band, product count tier, income band) |

## Prerequisites
- Notebook 01 must have run successfully.
- `ChurnAnalysisLH` must be attached as the Lakehouse.
- Tables: `customers`, `accounts`, `products`, `customer_products`, `transactions` must exist.

## Cell 1 — Imports and Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, when, lit, round as spark_round,
    coalesce, max as spark_max, min as spark_min
)

spark = SparkSession.builder.getOrCreate()

# Reference date — must match Notebook 01
from datetime import date, timedelta
REFERENCE_DATE = date(2024, 3, 31)
DATE_90_DAYS_AGO = str(REFERENCE_DATE - timedelta(days=90))

print("✅ Spark session ready")
print(f"   Reference date: {REFERENCE_DATE}")
print(f"   90-day window start: {DATE_90_DAYS_AGO}")

## Cell 2 — Load Source Tables

In [ ]:
customers_df       = spark.read.table("customers")
accounts_df        = spark.read.table("accounts")
transactions_df    = spark.read.table("transactions")
customer_products_df = spark.read.table("customer_products")

print("✅ Source tables loaded:")
print(f"   customers:         {customers_df.count():>10,}")
print(f"   accounts:          {accounts_df.count():>10,}")
print(f"   transactions:      {transactions_df.count():>10,}")
print(f"   customer_products: {customer_products_df.count():>10,}")

## Cell 3 — Compute Account-Level Aggregates

For each customer, aggregate account information:
- Total and average balance across all accounts
- Dominant balance trend (most common trend across accounts)

In [ ]:
from pyspark.sql.functions import first

# Account-level aggregation per customer
account_agg = accounts_df.groupBy("customer_id").agg(
    count("account_id").alias("account_count"),
    spark_round(spark_sum("avg_balance_90d"), 2).alias("total_balance_90d"),
    spark_round(avg("avg_balance_90d"), 2).alias("avg_balance_90d"),
    spark_round(avg("current_balance"), 2).alias("avg_current_balance"),
    first("balance_trend").alias("balance_trend"),  # dominant trend
)

print(f"✅ Account aggregates: {account_agg.count():,} customers with account data")
account_agg.show(5)

## Cell 4 — Compute Product Counts per Customer

In [ ]:
product_agg = customer_products_df.groupBy("customer_id").agg(
    count("product_id").alias("total_product_count"),
    spark_sum("active_product_flag").alias("active_product_count"),
)

print(f"✅ Product aggregates: {product_agg.count():,} customers with product data")
product_agg.show(5)

## Cell 5 — Compute Transaction Metrics per Customer (Last 90 Days)

In [ ]:
from pyspark.sql.functions import to_date, abs as spark_abs

# Filter to last-90-day transactions
recent_txn = transactions_df.filter(
    col("transaction_date") >= DATE_90_DAYS_AGO
)

txn_agg = recent_txn.groupBy("customer_id").agg(
    count("transaction_id").alias("transaction_count_90d"),
    spark_round(
        spark_sum(when(col("amount") < 0, spark_abs(col("amount"))).otherwise(lit(0))),
        2
    ).alias("total_spend_90d"),
    spark_round(avg(spark_abs(col("amount"))), 2).alias("avg_txn_amount_90d"),
)

print(f"✅ Transaction aggregates: {txn_agg.count():,} customers with recent transactions")

## Cell 6 — Build Customer 360 Table

Join all aggregates to the customers base table, then apply segmentation logic.

In [ ]:
# Join all aggregates to customers
c360 = customers_df \
    .join(account_agg,  on="customer_id", how="left") \
    .join(product_agg,  on="customer_id", how="left") \
    .join(txn_agg,      on="customer_id", how="left")

# Fill nulls for customers with no accounts / products / transactions
c360 = c360 \
    .fillna({"account_count": 0, "total_balance_90d": 0.0, "avg_balance_90d": 0.0,
             "avg_current_balance": 0.0}) \
    .fillna({"total_product_count": 0, "active_product_count": 0}) \
    .fillna({"transaction_count_90d": 0, "total_spend_90d": 0.0, "avg_txn_amount_90d": 0.0}) \
    .fillna({"balance_trend": "Stable"})

print(f"Customer 360 before segmentation: {c360.count():,} rows")
c360.printSchema()

## Cell 7 — Apply Segmentation Logic

Add three segmentation dimensions based on clear, business-driven thresholds:

| Dimension | Logic |
|---|---|
| `activity_tier` | Based on transaction count in last 90 days |
| `balance_band` | Based on average balance in last 90 days |
| `product_count_tier` | Based on number of active products |

In [ ]:
# Activity tier segmentation
c360 = c360.withColumn(
    "activity_tier",
    when(col("transaction_count_90d") == 0, "Inactive")
    .when(col("transaction_count_90d") <= 3, "Low")
    .when(col("transaction_count_90d") <= 9, "Medium")
    .otherwise("High")
)

# Balance band segmentation (EUR)
c360 = c360.withColumn(
    "balance_band",
    when(col("avg_balance_90d") >= 25000, "High")
    .when(col("avg_balance_90d") >= 5000,  "Medium")
    .when(col("avg_balance_90d") >= 500,   "Low")
    .otherwise("Very Low")
)

# Product count tier segmentation
c360 = c360.withColumn(
    "product_count_tier",
    when(col("active_product_count") == 0, "No product")
    .when(col("active_product_count") == 1, "Single")
    .when(col("active_product_count") == 2, "Dual")
    .otherwise("Multi-product")
)

# Select final columns for customer_360
customer_360_df = c360.select(
    "customer_id",
    "age",
    "region",
    "tenure_months",
    "income_band",
    "risk_profile",
    "digital_active_flag",
    "churned_90d",
    "avg_balance_90d",
    "balance_trend",
    "active_product_count",
    "transaction_count_90d",
    "total_spend_90d",
    "activity_tier",
    "balance_band",
    "product_count_tier",
)

print(f"✅ Customer 360 final: {customer_360_df.count():,} rows")

# Segment distribution sanity check
print("\nActivity tier distribution:")
customer_360_df.groupBy("activity_tier").count().orderBy("count", ascending=False).show()

print("Balance band distribution:")
customer_360_df.groupBy("balance_band").count().orderBy("count", ascending=False).show()

## Cell 8 — Compute Churn KPIs by Segment

Build the `churn_by_segment` table — aggregated churn KPIs for each segmentation dimension.

In [ ]:
from pyspark.sql import Row

def compute_segment_kpis(df, dimension_col, dimension_name):
    """Compute churn KPIs for a given segmentation column."""
    return df.groupBy(dimension_col).agg(
        count("customer_id").alias("total_customers"),
        spark_sum("churned_90d").alias("churned_customers"),
        spark_round(100.0 * spark_sum("churned_90d") / count("customer_id"), 2).alias("churn_rate_pct"),
        spark_round(avg("avg_balance_90d"), 2).alias("avg_balance"),
        spark_round(avg("active_product_count"), 2).alias("avg_product_count"),
        spark_round(avg("transaction_count_90d"), 2).alias("avg_transaction_count"),
    ).withColumn("segment_dimension", lit(dimension_name)) \
     .withColumnRenamed(dimension_col, "segment_value")

# Compute for each dimension
seg_activity = compute_segment_kpis(customer_360_df, "activity_tier",     "activity_tier")
seg_balance  = compute_segment_kpis(customer_360_df, "balance_band",      "balance_band")
seg_products = compute_segment_kpis(customer_360_df, "product_count_tier", "product_count_tier")
seg_income   = compute_segment_kpis(customer_360_df, "income_band",        "income_band")

# Union all segments
churn_by_segment_df = seg_activity.unionByName(seg_balance) \
    .unionByName(seg_products) \
    .unionByName(seg_income) \
    .select(
        "segment_dimension",
        "segment_value",
        "total_customers",
        "churned_customers",
        "churn_rate_pct",
        "avg_balance",
        "avg_product_count",
        "avg_transaction_count",
    )

print(f"✅ Churn by segment: {churn_by_segment_df.count():,} segment rows")
churn_by_segment_df.orderBy("churn_rate_pct", ascending=False).show(20)

## Cell 9 — Write Output Delta Tables

In [ ]:
print("Writing curated Delta tables...")

customer_360_df.write.format("delta").mode("overwrite").saveAsTable("customer_360")
print("  ✅ Delta table: customer_360")

churn_by_segment_df.write.format("delta").mode("overwrite").saveAsTable("churn_by_segment")
print("  ✅ Delta table: churn_by_segment")

print("\n✅ All curated tables written.")

## Cell 10 — Validation and Business Summary

Final sanity checks and a summary of key churn insights.

In [ ]:
print("=" * 60)
print("VALIDATION — Final Table Counts")
print("=" * 60)

for tname in ["customers", "accounts", "products", "customer_products",
               "transactions", "customer_360", "churn_by_segment"]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {tname}").collect()[0][0]
    print(f"  {tname:<30} {n:>10,}")

print()
print("=" * 60)
print("BUSINESS SUMMARY — Churn Insights")
print("=" * 60)

# Overall KPI
spark.sql("""
    SELECT
        COUNT(*) AS total_customers,
        SUM(churned_90d) AS churned,
        ROUND(100.0 * SUM(churned_90d) / COUNT(*), 2) AS overall_churn_rate_pct
    FROM customer_360
""").show()

# Top churn segments
print("Top segments by churn rate (min 50 customers):")
spark.sql("""
    SELECT segment_dimension, segment_value, total_customers, churned_customers, churn_rate_pct
    FROM churn_by_segment
    WHERE total_customers >= 50
    ORDER BY churn_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

print("✅ Notebook 02 complete.")
print("   Next steps:")
print("   → Run SQL queries from sql/02_descriptive_churn_queries.sql")
print("   → Build Power BI report (see powerbi/report_design.md)")
print("   → Configure Data Agent (see docs/data_agent_setup.md)")

## Cell 11 — Visualisations de l'analyse de churn

Pour compléter le rapport Power BI, cette section produit une série de graphiques directement dans le notebook afin d'illustrer l'analyse de churn : taux de churn global, taux de churn par segment, et distributions des variables clés selon le statut de churn.

> Les DataFrames Spark sont convertis en pandas (`.toPandas()`) uniquement pour la visualisation — les volumes agrégés restent faibles.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    plt.style.use("seaborn-v0_8-whitegrid")

# Palette Contoso Banque
CHURN_COLORS = {0: "#2E7D32", 1: "#C62828"}   # vert = retenu, rouge = churné
ACCENT = "#00754A"

# Conversion en pandas pour la visualisation
pdf = customer_360_df.toPandas()
seg_pdf = churn_by_segment_df.toPandas()

pdf["churn_label"] = pdf["churned_90d"].map({0: "Retenu", 1: "Churné"})

print(f"✅ Données chargées pour visualisation : {len(pdf):,} clients")
print(f"   Seaborn disponible : {HAS_SEABORN}")


In [ ]:
# --- Vue d'ensemble : taux de churn global ---
counts = pdf["churn_label"].value_counts()
churn_rate = 100.0 * pdf["churned_90d"].sum() / len(pdf)

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=counts.index,
    autopct=lambda p: f"{p:.1f}%\n({int(round(p * len(pdf) / 100)):,})",
    colors=[CHURN_COLORS[0] if lbl == "Retenu" else CHURN_COLORS[1] for lbl in counts.index],
    startangle=90,
    wedgeprops=dict(width=0.42, edgecolor="white"),
    textprops=dict(fontsize=11),
)
plt.setp(autotexts, color="white", fontweight="bold")
ax.text(0, 0, f"{churn_rate:.1f}%\nchurn", ha="center", va="center",
        fontsize=18, fontweight="bold", color=CHURN_COLORS[1])
ax.set_title("Taux de churn global — Contoso Banque", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# --- Taux de churn par segment (les 4 dimensions) ---
dimension_titles = {
    "activity_tier":      "Niveau d'activité",
    "balance_band":       "Tranche de solde",
    "product_count_tier": "Nombre de produits",
    "income_band":        "Tranche de revenu",
}
# Ordre logique pour l'affichage
dimension_orders = {
    "activity_tier":      ["Inactive", "Low", "Medium", "High"],
    "balance_band":       ["Very Low", "Low", "Medium", "High"],
    "product_count_tier": ["No product", "Single", "Dual", "Multi-product"],
    "income_band":        None,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (dim, title) in enumerate(dimension_titles.items()):
    ax = axes[i]
    sub = seg_pdf[seg_pdf["segment_dimension"] == dim].copy()
    order = dimension_orders[dim]
    if order:
        sub["__ord"] = sub["segment_value"].apply(
            lambda v: order.index(v) if v in order else len(order))
        sub = sub.sort_values("__ord")
    else:
        sub = sub.sort_values("churn_rate_pct", ascending=False)

    bars = ax.bar(sub["segment_value"], sub["churn_rate_pct"], color=ACCENT, edgecolor="white")
    ax.axhline(churn_rate, color=CHURN_COLORS[1], linestyle="--", linewidth=1.2,
               label=f"Moyenne globale ({churn_rate:.1f}%)")
    ax.set_title(f"Taux de churn — {title}", fontsize=12, fontweight="bold")
    ax.set_ylabel("Taux de churn (%)")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8)
    for b, val in zip(bars, sub["churn_rate_pct"]):
        ax.text(b.get_x() + b.get_width() / 2, val + 0.3, f"{val:.1f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

fig.suptitle("Taux de churn par segment", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
# --- Distributions des variables clés selon le statut de churn ---
numeric_features = [
    ("avg_balance_90d",       "Solde moyen 90j (EUR)"),
    ("transaction_count_90d", "Nb transactions 90j"),
    ("tenure_months",         "Ancienneté (mois)"),
    ("age",                   "Âge"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, (colname, label) in enumerate(numeric_features):
    ax = axes[i]
    retained = pdf.loc[pdf["churned_90d"] == 0, colname].dropna()
    churned  = pdf.loc[pdf["churned_90d"] == 1, colname].dropna()

    # Clip pour limiter l'effet des valeurs extrêmes sur l'axe (solde surtout)
    if colname == "avg_balance_90d":
        upper = pdf[colname].quantile(0.98)
        retained = retained.clip(upper=upper)
        churned = churned.clip(upper=upper)

    if HAS_SEABORN:
        sns.histplot(retained, ax=ax, color=CHURN_COLORS[0], label="Retenu",
                     stat="density", kde=True, alpha=0.45, bins=30)
        sns.histplot(churned, ax=ax, color=CHURN_COLORS[1], label="Churné",
                     stat="density", kde=True, alpha=0.45, bins=30)
    else:
        ax.hist(retained, bins=30, density=True, color=CHURN_COLORS[0], alpha=0.5, label="Retenu")
        ax.hist(churned, bins=30, density=True, color=CHURN_COLORS[1], alpha=0.5, label="Churné")

    ax.set_title(f"Distribution — {label}", fontsize=12, fontweight="bold")
    ax.set_xlabel(label)
    ax.set_ylabel("Densité")
    ax.legend(fontsize=9)

fig.suptitle("Distributions des variables clés : clients retenus vs churnés", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
# --- Boxplots comparatifs par statut de churn ---
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for i, (colname, label) in enumerate(numeric_features):
    ax = axes[i]
    data = [
        pdf.loc[pdf["churned_90d"] == 0, colname].dropna(),
        pdf.loc[pdf["churned_90d"] == 1, colname].dropna(),
    ]
    bp = ax.boxplot(data, labels=["Retenu", "Churné"], patch_artist=True, showfliers=False)
    for patch, cval in zip(bp["boxes"], [CHURN_COLORS[0], CHURN_COLORS[1]]):
        patch.set_facecolor(cval)
        patch.set_alpha(0.6)
    for median in bp["medians"]:
        median.set_color("black")
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_ylabel(label)

fig.suptitle("Comparaison des variables clés par statut de churn (boxplots)", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# --- Heatmap croisée : taux de churn Activité x Solde + churn par région ---
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1) Heatmap activity_tier x balance_band
act_order = ["Inactive", "Low", "Medium", "High"]
bal_order = ["Very Low", "Low", "Medium", "High"]
pivot = (
    pdf.groupby(["activity_tier", "balance_band"])["churned_90d"]
    .mean().mul(100).reset_index()
    .pivot(index="activity_tier", columns="balance_band", values="churned_90d")
    .reindex(index=act_order, columns=bal_order)
)

if HAS_SEABORN:
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Reds", ax=ax1,
                cbar_kws={"label": "Taux de churn (%)"}, linewidths=0.5, linecolor="white")
else:
    im = ax1.imshow(pivot.values, cmap="Reds", aspect="auto")
    ax1.set_xticks(range(len(bal_order))); ax1.set_xticklabels(bal_order)
    ax1.set_yticks(range(len(act_order))); ax1.set_yticklabels(act_order)
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            v = pivot.values[r, c]
            if not np.isnan(v):
                ax1.text(c, r, f"{v:.1f}", ha="center", va="center")
    fig.colorbar(im, ax=ax1, label="Taux de churn (%)")

ax1.set_title("Taux de churn — Activité x Solde", fontsize=12, fontweight="bold")
ax1.set_xlabel("Tranche de solde")
ax1.set_ylabel("Niveau d'activité")

# 2) Taux de churn par région
region = (
    pdf.groupby("region")["churned_90d"].agg(["mean", "count"]).reset_index()
    .rename(columns={"mean": "churn_rate", "count": "n"})
)
region["churn_rate"] *= 100
region = region.sort_values("churn_rate", ascending=False)

bars = ax2.barh(region["region"], region["churn_rate"], color=ACCENT, edgecolor="white")
ax2.axvline(churn_rate, color=CHURN_COLORS[1], linestyle="--", linewidth=1.2,
            label=f"Moyenne globale ({churn_rate:.1f}%)")
ax2.set_title("Taux de churn par région", fontsize=12, fontweight="bold")
ax2.set_xlabel("Taux de churn (%)")
ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
ax2.invert_yaxis()
ax2.legend(fontsize=9)
for b, val, n in zip(bars, region["churn_rate"], region["n"]):
    ax2.text(val + 0.2, b.get_y() + b.get_height() / 2, f"{val:.1f}% (n={n:,})",
             va="center", fontsize=9)

plt.tight_layout()
plt.show()

print("✅ Visualisations générées. Ces graphiques complètent le rapport Power BI.")
